In [ ]:

from pyspark.sql import SparkSession
#from pyspark.sql.functions import col, explode, lit

In [ ]:
spark = SparkSession.builder \
        .appName("Complex_json") \
        .getOrCreate()

In [ ]:
file_path = "/home/akshay/Documents/Setups/1Projects/RetailAnalysis/misc/PySpark/data/medications.json"

In [ ]:
# Read the multi-line JSON document and inspect its nested schema.
raw_df = spark.read.option("multiLine", "true").json(file_path)
raw_df.printSchema()
raw_df.show(truncate=False)



In [ ]:
# Explode the lab and imaging arrays into one row per item.
labs_df = raw_df.select(explode(col("labs")).alias("lab")).select("lab.*")
imaging_df = raw_df.select(explode(col("imaging")).alias("image")).select("image.*")



In [ ]:
# Each medication category is an array inside the medications array.
medication_groups_df = raw_df.select(
    explode(col("medications")).alias("medication_group")
)
medication_categories = (
    raw_df.schema["medications"].dataType.elementType.fieldNames()
)

medication_dfs = []
for category in medication_categories:
    category_df = (
        medication_groups_df
        .select(
            explode(col(f"medication_group.{category}")).alias("medication")
        )
        .select(lit(category).alias("category"), col("medication.*"))
    )
    medication_dfs.append(category_df)

medications_df = medication_dfs[0]
for category_df in medication_dfs[1:]:
    medications_df = medications_df.unionByName(category_df)



In [ ]:
print("Medications:")
medications_df.show(truncate=False)

In [ ]:
print("Labs:")
labs_df.show(truncate=False)

In [ ]:
print("Imaging:")
imaging_df.show(truncate=False)

In [ ]:
spark.stop()